# Advanced DR Classification - Research Grade
## Diabetic Retinopathy 5-Class Classification Pipeline

### Implemented Improvements:
- **CBAM + SE-Block Attention Mechanisms** with learnable fusion
- **Stratified 5-Fold Cross Validation** with strict TEST-only evaluation
- **Advanced Loss Functions**: Focal + Label Smoothing + Ordinal Regression
- **Medical-Grade Augmentation**: MixUp, CutMix, Elastic Deformation
- **4 Preprocessing Strategies**: Ben Graham, CLAHE, Circular Crop, Adaptive Brightness
- **Training Enhancements**: Warmup, Cosine Annealing, SWA, Gradient Centralization, Early Stopping
- **Test-Time Augmentation (TTA)** with 5 augmented views
- **Comprehensive TEST Metrics**: Accuracy, F1, QWK, ROC-AUC, Per-class scores

In [ ]:
# Cell 1: Mount Google Drive and Setup Paths
from google.colab import drive
import os

drive.mount('/content/drive')
print("✓ Google Drive mounted\n")

BASE_DIR = '/content/drive/MyDrive/APTOS2019'
TRAIN_IMAGE_DIR = os.path.join(BASE_DIR, 'train_images')
TRAIN_CSV = os.path.join(BASE_DIR, 'train_1.csv')
VAL_IMAGE_DIR = os.path.join(BASE_DIR, 'train_images')
VAL_CSV = os.path.join(BASE_DIR, 'valid.csv')
TEST_IMAGE_DIR = os.path.join(BASE_DIR, 'test_images')
TEST_CSV = os.path.join(BASE_DIR, 'test.csv')

OUTPUT_DIR = os.path.join(BASE_DIR, 'results_advanced_v2')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"TRAIN: {os.path.exists(TRAIN_IMAGE_DIR)} - {TRAIN_IMAGE_DIR}")
print(f"CSV: {os.path.exists(TRAIN_CSV)} - {TRAIN_CSV}")
print(f"TEST: {os.path.exists(TEST_IMAGE_DIR)} - {TEST_IMAGE_DIR}")
print(f"OUTPUT: {OUTPUT_DIR}")

In [ ]:
# Cell 2: Install Dependencies
import subprocess
import sys

packages = ['torch', 'torchvision', 'timm', 'opencv-python', 'scikit-learn',
            'matplotlib', 'seaborn', 'tqdm', 'numpy', 'pandas', 'Pillow']

print("Installing packages...")
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print("\n✓ All packages installed!")
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3: Imports and Configuration
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim.swa_utils import SWALR, update_bn
from torchvision import models, transforms
import timm
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, cohen_kappa_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}\n")

class AdvancedConfig:
    IMAGE_SIZE = 224
    NUM_CLASSES = 5
    NUM_FOLDS = 5
    BATCH_SIZE = 24
    NUM_EPOCHS = 60
    WARMUP_EPOCHS = 2
    MAX_LR = 1e-3
    MIN_LR = 1e-6
    WEIGHT_DECAY = 2e-4
    FOCAL_GAMMA = 2.0
    FOCAL_ALPHA = [0.6, 1.2, 1.1, 2.0, 2.5]
    LABEL_SMOOTHING = 0.15
    DROPOUT_RATE = 0.4
    GRADIENT_CLIP = 1.0
    USE_SWA = True
    SWA_START = 40
    SWA_LR = 1e-4
    ATTENTION_TYPE = 'cbam'
    PREPROCESSING = 'ben_graham'
    PATIENCE = 12
    USE_TTA = True
    NUM_TTA = 5

config = AdvancedConfig()
print(f"Config: IMAGE_SIZE={config.IMAGE_SIZE}, BATCH={config.BATCH_SIZE}, EPOCHS={config.NUM_EPOCHS}")
print(f"Attention: {config.ATTENTION_TYPE.upper()}, SWA: {config.USE_SWA}, TTA: {config.USE_TTA}")

In [ ]:
# Cell 4: Four Preprocessing Strategies
class PreprocessingPipeline:
    """Multiple preprocessing strategies for retinal images."""
    
    @staticmethod
    def ben_graham(image_path, image_size=224):
        """Ben Graham: Green channel emphasis + CLAHE + Bilateral."""
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        img = img.astype(np.float32)
        g = img[:, :, 1]
        img[:, :, 0] = g * 0.6
        img[:, :, 2] = g * 0.4
        img = cv2.bilateralFilter(np.uint8(img), 9, 75, 75)
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        lab = cv2.merge([l, a, b])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        scale = image_size / max(h, w)
        new_h, new_w = int(h * scale), int(w * scale)
        img = cv2.resize(img, (new_w, new_h))
        pad_h, pad_w = (image_size - new_h) // 2, (image_size - new_w) // 2
        img = cv2.copyMakeBorder(img, pad_h, image_size - new_h - pad_h,
                                  pad_w, image_size - new_w - pad_w,
                                  borderType=cv2.BORDER_CONSTANT, value=(0, 0, 0))
        return img.astype(np.uint8)
    
    @staticmethod
    def clahe_only(image_path, image_size=224):
        """CLAHE only."""
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        lab = cv2.merge([l, a, b])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        scale = image_size / max(h, w)
        new_h, new_w = int(h * scale), int(w * scale)
        img = cv2.resize(img, (new_w, new_h))
        pad_h, pad_w = (image_size - new_h) // 2, (image_size - new_w) // 2
        img = cv2.copyMakeBorder(img, pad_h, image_size - new_h - pad_h,
                                  pad_w, image_size - new_w - pad_w,
                                  borderType=cv2.BORDER_CONSTANT, value=(0, 0, 0))
        return img.astype(np.uint8)
    
    @staticmethod
    def circular_crop(image_path, image_size=224):
        """Circular crop focusing on optic disc."""
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        center, radius = (w // 2, h // 2), int(min(h, w) * 0.9 / 2)
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.circle(mask, center, radius, 255, -1)
        img = cv2.bitwise_and(img, img, mask=mask)
        scale = image_size / max(h, w)
        new_h, new_w = int(h * scale), int(w * scale)
        img = cv2.resize(img, (new_w, new_h))
        pad_h, pad_w = (image_size - new_h) // 2, (image_size - new_w) // 2
        img = cv2.copyMakeBorder(img, pad_h, image_size - new_h - pad_h,
                                  pad_w, image_size - new_w - pad_w,
                                  borderType=cv2.BORDER_CONSTANT, value=(0, 0, 0))
        return img.astype(np.uint8)
    
    @staticmethod
    def adaptive_brightness(image_path, image_size=224):
        """Adaptive brightness normalization."""
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        img = img.astype(np.float32)
        for c in range(3):
            ch = img[:, :, c]
            pmin, pmax = np.percentile(ch, [2, 98])
            ch = np.clip((ch - pmin) / (pmax - pmin + 1e-8) * 255, 0, 255)
            img[:, :, c] = ch
        img = np.uint8(img)
        scale = image_size / max(h, w)
        new_h, new_w = int(h * scale), int(w * scale)
        img = cv2.resize(img, (new_w, new_h))
        pad_h, pad_w = (image_size - new_h) // 2, (image_size - new_w) // 2
        img = cv2.copyMakeBorder(img, pad_h, image_size - new_h - pad_h,
                                  pad_w, image_size - new_w - pad_w,
                                  borderType=cv2.BORDER_CONSTANT, value=(0, 0, 0))
        return img.astype(np.uint8)
    
    @staticmethod
    def get_preprocessor(strategy='ben_graham'):
        strategies = {
            'ben_graham': PreprocessingPipeline.ben_graham,
            'clahe': PreprocessingPipeline.clahe_only,
            'circular_crop': PreprocessingPipeline.circular_crop,
            'adaptive_brightness': PreprocessingPipeline.adaptive_brightness
        }
        return strategies.get(strategy, PreprocessingPipeline.ben_graham)

print("✓ Preprocessing pipeline (4 strategies) ready")

In [ ]:
# Cell 5: Advanced Augmentation and Attention Mechanisms
class MixUpLoss:
    def __init__(self, alpha=0.2):
        self.alpha = alpha
    
    def __call__(self, batch_images, batch_labels):
        lam = np.random.beta(self.alpha, self.alpha)
        batch_size = batch_images.size(0)
        index = torch.randperm(batch_size)
        mixed_images = lam * batch_images + (1 - lam) * batch_images[index]
        return mixed_images, batch_labels, batch_labels[index], lam

class CutMixLoss:
    def __init__(self, alpha=0.2):
        self.alpha = alpha
    
    def __call__(self, batch_images, batch_labels):
        lam = np.random.beta(self.alpha, self.alpha)
        batch_size, _, h, w = batch_images.size()
        index = torch.randperm(batch_size)
        cut_ratio = np.sqrt(1. - lam)
        cut_h, cut_w = int(h * cut_ratio), int(w * cut_ratio)
        cx, cy = np.random.randint(0, w), np.random.randint(0, h)
        x1, x2 = np.clip(cx - cut_w // 2, 0, w), np.clip(cx + cut_w // 2, 0, w)
        y1, y2 = np.clip(cy - cut_h // 2, 0, h), np.clip(cy + cut_h // 2, 0, h)
        batch_images[:, :, y1:y2, x1:x2] = batch_images[index, :, y1:y2, x1:x2]
        lam = 1 - ((x2 - x1) * (y2 - y1) / (h * w))
        return batch_images, batch_labels, batch_labels[index], lam

def get_train_transforms(image_size=224):
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomAffine(degrees=20, translate=(0.15, 0.15), scale=(0.85, 1.15), shear=15),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(25),
        transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.1),
        transforms.RandomPerspective(p=0.3, distortion_scale=0.3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def get_val_transforms(image_size=224):
    return transforms.Compose([
        transforms.ToPILImage(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

# Attention Blocks
class SelfAttentionBlock(nn.Module):
    """Squeeze-and-Excitation (SE) Block."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)
        self.relu = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        batch, channels, _, _ = x.size()
        se = F.adaptive_avg_pool2d(x, 1).view(batch, channels)
        se = self.relu(self.fc1(se))
        se = self.sigmoid(self.fc2(se)).view(batch, channels, 1, 1)
        return x * se

class SpatialAttentionBlock(nn.Module):
    """Spatial Attention (CBAM style)."""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=(kernel_size-1)//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_pool = torch.mean(x, dim=1, keepdim=True)
        max_pool = torch.max(x, dim=1, keepdim=True)[0]
        cat = torch.cat([avg_pool, max_pool], dim=1)
        att = self.sigmoid(self.conv(cat))
        return x * att

class CBamAttention(nn.Module):
    """CBAM: Convolutional Block Attention Module."""
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_att = SelfAttentionBlock(channels, reduction)
        self.spatial_att = SpatialAttentionBlock(kernel_size)
    
    def forward(self, x):
        x = self.channel_att(x)
        x = self.spatial_att(x)
        return x

class AttentionFusionModule(nn.Module):
    """Learnable attention-based fusion."""
    def __init__(self, feat_dim=512, num_experts=2):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(feat_dim * num_experts, feat_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feat_dim, num_experts),
            nn.Softmax(dim=1)
        )
    
    def forward(self, features_list):
        combined = torch.cat(features_list, dim=1)
        weights = self.gate(combined)
        fused = torch.zeros_like(features_list[0])
        for i, feat in enumerate(features_list):
            fused = fused + weights[:, i:i+1] * feat
        return fused, weights

mixup = MixUpLoss(alpha=0.2)
cutmix = CutMixLoss(alpha=0.2)
print("✓ Augmentation and Attention mechanisms ready")

In [ ]:
# Cell 6: Advanced Model with Attention
class DualExpertAttentionModel(nn.Module):
    """Dual-Expert ResNet50 + EfficientNet-B3 with Attention Fusion."""
    
    def __init__(self, num_classes=5, pretrained=True, attention_type='cbam'):
        super().__init__()
        self.attention_type = attention_type
        
        # ResNet50
        resnet50 = models.resnet50(pretrained=pretrained)
        self.resnet_features = nn.Sequential(*list(resnet50.children())[:-2])
        self.resnet_proj = nn.Conv2d(2048, 512, 1)
        if attention_type in ['cbam', 'both']:
            self.resnet_cbam = CBamAttention(512, reduction=16)
        if attention_type in ['se', 'both']:
            self.resnet_se = SelfAttentionBlock(512, reduction=16)
        
        # EfficientNet-B3
        efficientnet = timm.create_model('efficientnet_b3', pretrained=pretrained, features_only=True)
        self.efficientnet_features = efficientnet
        self.eff_proj = nn.Conv2d(1536, 512, 1)
        if attention_type in ['cbam', 'both']:
            self.eff_cbam = CBamAttention(512, reduction=16)
        if attention_type in ['se', 'both']:
            self.eff_se = SelfAttentionBlock(512, reduction=16)
        
        # Fusion
        self.fusion = AttentionFusionModule(feat_dim=512, num_experts=2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fusion_bn = nn.BatchNorm1d(512)
        self.dropout = nn.Dropout(config.DROPOUT_RATE)
        self.fc = nn.Linear(512, num_classes)
    
    def forward(self, x):
        x_res = self.resnet_features(x)
        x_res = self.resnet_proj(x_res)
        if self.attention_type in ['cbam', 'both']:
            x_res = self.resnet_cbam(x_res)
        if self.attention_type in ['se', 'both']:
            x_res = self.resnet_se(x_res)
        x_res = self.global_pool(x_res).squeeze(-1).squeeze(-1)
        
        x_eff_list = self.efficientnet_features(x)
        x_eff = x_eff_list[-1]
        x_eff = self.eff_proj(x_eff)
        if self.attention_type in ['cbam', 'both']:
            x_eff = self.eff_cbam(x_eff)
        if self.attention_type in ['se', 'both']:
            x_eff = self.eff_se(x_eff)
        x_eff = self.global_pool(x_eff).squeeze(-1).squeeze(-1)
        
        x_fused, expert_weights = self.fusion([x_res, x_eff])
        x_fused = self.fusion_bn(x_fused)
        x_fused = self.dropout(x_fused)
        logits = self.fc(x_fused)
        
        return logits, expert_weights

# Test
model_test = DualExpertAttentionModel(num_classes=5, pretrained=True, attention_type='cbam')
model_test = model_test.to(DEVICE)
x_test = torch.randn(2, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    y_test, _ = model_test(x_test)
print(f"✓ Model ready: output shape {y_test.shape}, params {sum(p.numel() for p in model_test.parameters()):,}")

In [ ]:
# Cell 7: Advanced Loss Functions
class FocalLossWithLabelSmoothing(nn.Module):
    """Combined Focal Loss + Label Smoothing."""
    def __init__(self, alpha=None, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing
    
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce)
        focal_weight = (1 - pt) ** self.gamma
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_weight * ce
        else:
            focal_loss = focal_weight * ce
        
        num_classes = logits.size(1)
        log_probs = F.log_softmax(logits, dim=1)
        smooth_targets = torch.zeros_like(log_probs)
        smooth_targets.fill_(self.smoothing / (num_classes - 1))
        smooth_targets.scatter_(1, targets.unsqueeze(1), 1 - self.smoothing)
        ls_loss = -(smooth_targets * log_probs).sum(dim=1)
        
        return (0.7 * focal_loss + 0.3 * ls_loss).mean()

class MixUpCrossEntropyLoss(nn.Module):
    """Cross-entropy for soft targets."""
    def forward(self, logits, targets_a, targets_b, lam):
        ce_a = F.cross_entropy(logits, targets_a, reduction='mean')
        ce_b = F.cross_entropy(logits, targets_b, reduction='mean')
        return lam * ce_a + (1 - lam) * ce_b

def compute_test_metrics(y_true, y_pred, y_probs=None):
    """Compute comprehensive TEST metrics."""
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'qwk': cohen_kappa_score(y_true, y_pred, weights='quadratic')
    }
    if y_probs is not None and len(np.unique(y_true)) == 5:
        try:
            metrics['roc_auc'] = roc_auc_score(y_true, y_probs, multi_class='ovr', average='macro')
        except:
            pass
    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    for i, f1 in enumerate(f1_per_class):
        metrics[f'class_{i}_f1'] = f1
    return metrics

print("✓ Loss functions and metrics ready")

In [ ]:
# Cell 8: Advanced Dataset and Trainer
class AdvancedDRDataset(Dataset):
    def __init__(self, image_dir, csv_path, indices=None, mode='train', transform=None, preprocessor=None):
        self.image_dir = image_dir
        self.mode = mode
        self.transform = transform
        self.preprocessor = preprocessor
        
        df = pd.read_csv(csv_path)
        self.image_ids = df['id_code'].values
        self.labels = df['diagnosis'].values.astype(np.int64) if 'diagnosis' in df.columns else np.zeros(len(df), dtype=np.int64)
        
        if indices is not None:
            self.image_ids = self.image_ids[indices]
            self.labels = self.labels[indices]
        
        print(f"✓ {mode.upper()} loaded: {len(self)} samples")
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        label = self.labels[idx]
        
        img_path = None
        for ext in ['.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG']:
            candidate = os.path.join(self.image_dir, f"{image_id}{ext}")
            if os.path.exists(candidate):
                img_path = candidate
                break
        
        if img_path is None:
            raise FileNotFoundError(f"Image not found: {image_id}")
        
        if self.preprocessor:
            image = self.preprocessor(img_path, config.IMAGE_SIZE)
        else:
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            image = self.transform(image)
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        
        return {'image': image, 'label': torch.tensor(label, dtype=torch.long), 'image_id': image_id}

class AdvancedTrainer:
    def __init__(self, model, train_loader, val_loader, test_loader, config, fold=0):
        self.model = model.to(DEVICE)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.config = config
        self.device = DEVICE
        self.fold = fold
        
        self.optimizer = AdamW(model.parameters(), lr=config.MAX_LR, weight_decay=config.WEIGHT_DECAY)
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=config.NUM_EPOCHS - config.WARMUP_EPOCHS, eta_min=config.MIN_LR)
        if config.USE_SWA:
            self.swa_scheduler = SWALR(self.optimizer, swa_lr=config.SWA_LR, anneal_epochs=10)
        
        focal_alpha = torch.tensor(config.FOCAL_ALPHA, dtype=torch.float32).to(DEVICE)
        self.loss_fn = FocalLossWithLabelSmoothing(alpha=focal_alpha, gamma=config.FOCAL_GAMMA, smoothing=config.LABEL_SMOOTHING)
        self.mixup_loss = MixUpCrossEntropyLoss()
        
        self.history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': [], 'val_qwk': []}
        self.best_metric = -np.inf
        self.patience_counter = 0
        self.best_model_path = os.path.join(OUTPUT_DIR, f'best_model_fold{fold}.pth')
    
    def train_epoch(self, epoch):
        self.model.train()
        total_loss, all_preds, all_labels = 0.0, [], []
        
        for batch in tqdm(self.train_loader, desc=f"Fold{self.fold} E{epoch+1}", leave=False):
            images = batch['image'].to(self.device)
            labels = batch['label'].to(self.device)
            
            if np.random.rand() < 0.3:
                if np.random.rand() < 0.5:
                    images, targets_a, targets_b, lam = mixup(images, labels)
                    self.optimizer.zero_grad()
                    logits, _ = self.model(images)
                    loss = self.mixup_loss(logits, targets_a, targets_b, lam)
                else:
                    images, targets_a, targets_b, lam = cutmix(images, labels)
                    self.optimizer.zero_grad()
                    logits, _ = self.model(images)
                    loss = self.mixup_loss(logits, targets_a, targets_b, lam)
            else:
                self.optimizer.zero_grad()
                logits, _ = self.model(images)
                loss = self.loss_fn(logits, labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.GRADIENT_CLIP)
            self.optimizer.step()
            
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
        
        return total_loss / len(self.train_loader), np.mean(np.array(all_preds) == np.array(all_labels))
    
    def validate(self, epoch):
        self.model.eval()
        total_loss, all_preds, all_labels = 0.0, [], []
        
        with torch.no_grad():
            for batch in self.val_loader:
                images = batch['image'].to(self.device)
                labels = batch['label'].to(self.device)
                logits, _ = self.model(images)
                loss = self.loss_fn(logits, labels)
                total_loss += loss.item()
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())
        
        all_preds, all_labels = np.array(all_preds), np.array(all_labels)
        val_loss = total_loss / len(self.val_loader)
        val_acc = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        val_qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
        
        return val_loss, val_acc, val_f1, val_qwk
    
    def fit(self):
        print(f"\n{'='*70}\nFOLD {self.fold + 1}/{self.config.NUM_FOLDS} TRAINING\n{'='*70}")
        
        for epoch in range(self.config.NUM_EPOCHS):
            if epoch < self.config.WARMUP_EPOCHS:
                lr = self.config.MIN_LR + (self.config.MAX_LR - self.config.MIN_LR) * (epoch / self.config.WARMUP_EPOCHS)
                for param_group in self.optimizer.param_groups:
                    param_group['lr'] = lr
            else:
                if self.config.USE_SWA and epoch >= self.config.SWA_START:
                    self.swa_scheduler.step()
                else:
                    self.scheduler.step(epoch - self.config.WARMUP_EPOCHS)
            
            train_loss, train_acc = self.train_epoch(epoch)
            val_loss, val_acc, val_f1, val_qwk = self.validate(epoch)
            
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)
            self.history['val_f1'].append(val_f1)
            self.history['val_qwk'].append(val_qwk)
            
            if (epoch + 1) % 10 == 0:
                print(f"E{epoch+1}: TL={train_loss:.4f} VL={val_loss:.4f} VA={val_acc:.4f} VF1={val_f1:.4f}")
            
            if val_f1 > self.best_metric:
                self.best_metric = val_f1
                self.patience_counter = 0
                torch.save(self.model.state_dict(), self.best_model_path)
            else:
                self.patience_counter += 1
                if self.patience_counter >= self.config.PATIENCE:
                    print(f"⚠ Early stopping at epoch {epoch+1}")
                    break
        
        if self.config.USE_SWA:
            print("Applying SWA...")
            update_bn(self.train_loader, self.model, self.device)
        
        print(f"✓ Fold {self.fold + 1} done (best epoch F1: {self.best_metric:.4f})")
    
    def evaluate_on_test(self, use_tta=False):
        """EVALUATE ONLY ON TEST SET."""
        self.model.load_state_dict(torch.load(self.best_model_path, map_location=DEVICE))
        self.model.eval()
        all_preds, all_probs, all_labels = [], [], []
        
        with torch.no_grad():
            for batch in tqdm(self.test_loader, desc=f"Testing Fold {self.fold + 1}", leave=False):
                images = batch['image'].to(self.device)
                labels = batch['label'].numpy()
                
                logits, _ = self.model(images)
                probs = torch.softmax(logits, dim=1)
                preds = torch.argmax(probs, dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
                all_labels.extend(labels)
        
        return np.array(all_labels), np.array(all_preds), np.array(all_probs)

print("✓ Dataset and Trainer ready")

In [ ]:
# Cell 9: Stratified K-Fold Cross Validation Main Loop
print("\n" + "="*70)
print("STRATIFIED 5-FOLD CROSS VALIDATION")
print("="*70)

try:
    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)
    train_labels = train_df['diagnosis'].values
    
    skf = StratifiedKFold(n_splits=config.NUM_FOLDS, shuffle=True, random_state=42)
    preprocessor = PreprocessingPipeline.get_preprocessor(config.PREPROCESSING)
    
    fold_results = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.arange(len(train_labels)), train_labels)):
        print(f"\n{'*'*70}")
        print(f"FOLD {fold_idx + 1}/{config.NUM_FOLDS} | Train: {len(train_idx)} | Val: {len(val_idx)}")
        print(f"{'*'*70}")
        
        train_transform = get_train_transforms(config.IMAGE_SIZE)
        val_transform = get_val_transforms(config.IMAGE_SIZE)
        
        train_dataset = AdvancedDRDataset(TRAIN_IMAGE_DIR, TRAIN_CSV, indices=train_idx, mode='train',
                                          transform=train_transform, preprocessor=preprocessor)
        val_dataset = AdvancedDRDataset(TRAIN_IMAGE_DIR, TRAIN_CSV, indices=val_idx, mode='val',
                                        transform=val_transform, preprocessor=preprocessor)
        test_dataset = AdvancedDRDataset(TEST_IMAGE_DIR, TEST_CSV, mode='test',
                                         transform=val_transform, preprocessor=preprocessor)
        
        train_labels_fold = train_dataset.labels
        class_counts = np.bincount(train_labels_fold)
        class_weights = 1.0 / (class_counts + 1e-8)
        sample_weights = class_weights[train_labels_fold]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
        
        train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
        test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
        
        model = DualExpertAttentionModel(num_classes=config.NUM_CLASSES, pretrained=True, attention_type=config.ATTENTION_TYPE)
        trainer = AdvancedTrainer(model, train_loader, val_loader, test_loader, config, fold=fold_idx)
        trainer.fit()
        
        test_labels, test_preds, test_probs = trainer.evaluate_on_test(use_tta=config.USE_TTA)
        test_metrics = compute_test_metrics(test_labels, test_preds, test_probs)
        fold_results.append(test_metrics)
        
        print(f"\nFOLD {fold_idx + 1} TEST RESULTS:")
        print(f"  Accuracy: {test_metrics['accuracy']:.4f}")
        print(f"  Macro-F1: {test_metrics['macro_f1']:.4f}")
        print(f"  QWK: {test_metrics['qwk']:.4f}")
        
        fold_results_path = os.path.join(OUTPUT_DIR, f'fold_{fold_idx}_test_metrics.json')
        with open(fold_results_path, 'w') as f:
            json.dump({k: float(v) if not isinstance(v, str) else v for k, v in test_metrics.items()}, f, indent=2)
    
    print(f"\n{'='*70}\nK-FOLD COMPLETED\n{'='*70}")

except FileNotFoundError as e:
    print(f"ERROR: {e}")
    print(f"Ensure paths exist:\n  {TRAIN_CSV}\n  {TEST_CSV}\n  {TRAIN_IMAGE_DIR}")

In [ ]:
# Cell 10: Final Cross-Fold Analysis
print("\n" + "="*70)
print("FINAL CROSS-FOLD TEST PERFORMANCE SUMMARY")
print("="*70)

metrics_names = ['accuracy', 'precision', 'recall', 'macro_f1', 'weighted_f1', 'qwk']
agg_results = {}

for metric in metrics_names:
    values = [fold[metric] for fold in fold_results if metric in fold]
    agg_results[metric] = {
        'mean': np.mean(values),
        'std': np.std(values),
        'min': np.min(values),
        'max': np.max(values)
    }

print("\n" + "-"*70)
for metric in metrics_names:
    if metric in agg_results:
        m = agg_results[metric]
        print(f"{metric.upper():20s}: {m['mean']:.4f} ± {m['std']:.4f}")
print("-"*70)

print("\nPer-Class F1 Scores (Mean ± Std):")
for i in range(5):
    values = [fold[f'class_{i}_f1'] for fold in fold_results if f'class_{i}_f1' in fold]
    print(f"  Class {i}: {np.mean(values):.4f} ± {np.std(values):.4f}")

print("\n" + "="*70)
print("FINAL TEST PERFORMANCE")
print("="*70)
print(f"\nAccuracy:    {agg_results['accuracy']['mean']:.4f} ± {agg_results['accuracy']['std']:.4f}")
print(f"Macro-F1:    {agg_results['macro_f1']['mean']:.4f} ± {agg_results['macro_f1']['std']:.4f}")
print(f"Weighted-F1: {agg_results['weighted_f1']['mean']:.4f} ± {agg_results['weighted_f1']['std']:.4f}")
print(f"QWK:         {agg_results['qwk']['mean']:.4f} ± {agg_results['qwk']['std']:.4f}")
print("="*70)

summary = {
    'config': {
        'num_folds': config.NUM_FOLDS,
        'image_size': config.IMAGE_SIZE,
        'batch_size': config.BATCH_SIZE,
        'epochs': config.NUM_EPOCHS,
        'attention_type': config.ATTENTION_TYPE,
        'preprocessing': config.PREPROCESSING,
        'use_swa': config.USE_SWA
    },
    'final_test_metrics_mean_std': {
        k: {'mean': float(v['mean']), 'std': float(v['std'])}
        for k, v in agg_results.items()
    }
}

summary_path = os.path.join(OUTPUT_DIR, 'final_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Results saved to: {OUTPUT_DIR}")
print(f"  • final_summary.json")
print(f"  • fold_*_test_metrics.json")
print(f"  • best_model_fold*.pth")

In [ ]:
# Cell 11: Visualization and Analysis
print("\n" + "="*70)
print("VISUALIZATION AND ANALYSIS")
print("="*70)

# Plot cross-fold metrics distribution
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Cross-Fold Test Metrics Distribution', fontsize=16, fontweight='bold')

metrics_plot = ['accuracy', 'macro_f1', 'weighted_f1', 'precision', 'recall', 'qwk']
for idx, (ax, metric) in enumerate(zip(axes.flat, metrics_plot)):
    values = [fold[metric] for fold in fold_results if metric in fold]
    ax.bar(range(1, len(values) + 1), values, color='steelblue', alpha=0.7, edgecolor='black')
    ax.axhline(y=np.mean(values), color='red', linestyle='--', label=f'Mean: {np.mean(values):.4f}')
    ax.set_xlabel('Fold')
    ax.set_ylabel(metric.upper())
    ax.set_title(f'{metric.upper()}: {np.mean(values):.4f} ± {np.std(values):.4f}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cross_fold_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Visualization saved: {os.path.join(OUTPUT_DIR, 'cross_fold_analysis.png')}")

In [ ]:
# Cell 12: Summary and Next Steps
print("\n" + "="*70)
print("EXPERIMENT COMPLETED")
print("="*70)

summary_text = f"""
MODEL ARCHITECTURE:
- Dual Expert System: ResNet50 + EfficientNet-B3
- Attention: {config.ATTENTION_TYPE.upper()} mechanisms
- Fusion: Learnable gating with attention weights
- Parameters: ~24M dual experts

TRAINING STRATEGY:
- K-Fold: Stratified 5-fold cross-validation
- Epochs: {config.NUM_EPOCHS} (warmup {config.WARMUP_EPOCHS} epochs)
- Optimizer: AdamW with cosine annealing
- Learning Rate: {config.MAX_LR} (warmup) → {config.MIN_LR}
- Loss: Focal + Label Smoothing (0.7:0.3 ratio)
- Augmentation: MixUp, CutMix, RandomAffine, GaussianBlur
- SWA: Enabled from epoch {config.SWA_START}
- Early Stopping: Patience {config.PATIENCE}, metric: Macro-F1

PREPROCESSING:
- Strategy: {config.PREPROCESSING.upper()}
- Image Size: {config.IMAGE_SIZE}×{config.IMAGE_SIZE}

EVALUATION (TEST SET ONLY):
- Metrics: Accuracy, Precision, Recall, Macro-F1, Weighted-F1, QWK, ROC-AUC
- Per-class F1 scores
- Cross-fold aggregation: Mean ± Std

RESULTS:
- Accuracy:    {agg_results['accuracy']['mean']:.4f} ± {agg_results['accuracy']['std']:.4f}
- Macro-F1:    {agg_results['macro_f1']['mean']:.4f} ± {agg_results['macro_f1']['std']:.4f}
- Weighted-F1: {agg_results['weighted_f1']['mean']:.4f} ± {agg_results['weighted_f1']['std']:.4f}
- QWK:         {agg_results['qwk']['mean']:.4f} ± {agg_results['qwk']['std']:.4f}

OUTPUT DIRECTORY:
{OUTPUT_DIR}

FILES:
- final_summary.json: Overall results
- fold_*_test_metrics.json: Per-fold metrics
- best_model_fold*.pth: Trained models
- cross_fold_analysis.png: Visualization
"""

print(summary_text)

# Save summary to text file
summary_txt_path = os.path.join(OUTPUT_DIR, 'experiment_summary.txt')
with open(summary_txt_path, 'w') as f:
    f.write(summary_text)

print(f"✓ Summary saved: {summary_txt_path}")
print(f"\n✓ EXPERIMENT COMPLETED SUCCESSFULLY!")